# Urdu Orpheus on Kaggle
Read README.md first. Configure GPU, Internet and RCLONE_CONFIG_B64 Secret. Replace REPO_URL, choose a fixed REPO_REF, and set session_hours below your displayed limit. Run a separate smoke experiment before the full epoch.
Use Save Version → Save & Run All for unattended execution.

Cell 1 — clone a fixed version

In [ ]:
import subprocess
from pathlib import Path
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPOSITORY.git"
REPO_REF = "main"  # Prefer a fixed tag/commit before starting the real study.
REPO = Path("/kaggle/working/urdu-orpheus")
if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "checkout", REPO_REF], cwd=REPO, check=True)


Cell 2 — install and preflight

In [ ]:
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO / "requirements-kaggle.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "check"], check=True)
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "rclone"], check=True)
subprocess.run([sys.executable, "-c", "from unsloth import FastLanguageModel; import torch; from snac import SNAC; assert torch.cuda.is_available(); assert tuple(map(int, torch.__version__.split('+')[0].split('.')[:2])) >= (2,6); print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))"], check=True)
subprocess.run(["df", "-h", "/kaggle/working"], check=True)


Cell 3 — private credentials and config

In [ ]:
import os, json
from kaggle_secrets import UserSecretsClient
os.environ["RCLONE_CONFIG_B64"] = UserSecretsClient().get_secret("RCLONE_CONFIG_B64")
os.environ["ORPHEUS_DRIVE_REMOTE"] = "gdrive:orpheus_urdu"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
cfg = json.loads((REPO / "configs/aslp50h.json").read_text())
cfg["session_hours"] = 10.5  # CHANGE to below the runtime limit displayed in your account.
CONFIG = Path("/kaggle/working/run-config.json")
CONFIG.write_text(json.dumps(cfg, indent=2))
RUN = Path(cfg["work_dir"]) / "runs" / cfg["run_id"]


Cell 4A — recommended unattended saved run

In [ ]:
# Blocks the notebook until training finishes or gracefully checkpoints for resume.
subprocess.run([sys.executable, "-u", str(REPO / "finetune_aslp_50h.py"),
                "--config", str(CONFIG)], cwd=REPO, check=True)


Cell 5 — status and latest samples

In [ ]:
from IPython.display import Audio, display
status = RUN / "status.json"
print(json.loads(status.read_text()) if status.exists() else "Preparing dataset/cache")
for path in sorted((RUN / "samples").glob("step-*/*.wav"))[-3:]:
    print(path)
    display(Audio(filename=str(path)))


Cell 6 — test the final adapter after completion

In [ ]:
status_value = json.loads((RUN / "status.json").read_text())
if status_value["status"] in ("complete", "smoke_complete"):
    output = Path("/kaggle/working/urdu_test.wav")
    subprocess.run([sys.executable, str(REPO / "synthesize.py"),
                    "--adapter", str(RUN / "adapter_final"),
                    "--text", "آج موسم بہت خوشگوار ہے اور ہم سب باہر سیر کے لیے جا رہے ہیں۔",
                    "--output", str(output)], cwd=REPO, check=True)
    display(Audio(filename=str(output)))
else:
    print("Saved for continuation; rerun the same config to finish the epoch.")
